# 🎙️ Deepfake Detection Pipeline

**Steps:**
1. Clone repo & install deps
2. Generate audio dataset (real + synthetic pairs)
3. Extract features (CQT or LFCC)

**Prerequisites:**
- Upload training data to Google Drive
- Know your GitHub repo URL

## 1️⃣ Setup & Clone

In [ ]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# --- CONFIGURATION ---
REPO_URL = "https://github.com/YOUR_USERNAME/DDAA-KAN.git" # <--- CHANGE THIS
# ---------------------

import os
if not os.path.exists("DDAA-KAN"):
    if "YOUR_USERNAME" in REPO_URL:
        print("⚠️ PLEASE UPDATE 'REPO_URL' ABOVE!")
    else:
        !git clone {REPO_URL}

if os.path.exists("DDAA-KAN"):
    %cd DDAA-KAN
    print("✅ Cloned successfully!")
else:
    print("❌ Clone failed (check URL)")

In [ ]:
# Install Deps
!apt-get install -y ffmpeg espeak-ng
!pip install -r requirements_colab.txt

## 2️⃣ Configure Data Sources

In [ ]:
import yaml

# --- PATHS CONFIG ---
DRIVE_DATA_PATH = "/content/drive/MyDrive/mozilla_cv_data/extracted/cv-corpus-24.0-2025-12-05/en"
OUTPUT_PATH = "/content/drive/MyDrive/DDAA_Pipeline_Output"
# --------------------

with open("config.yaml", "r") as f:
    config = yaml.safe_load(f)

# Enable Synthesis
config['synthesis']['pick_strategy'] = 'random'
config['synthesis']['tts_models'] = ["tts_models/en/ljspeech/vits"]
config['synthesis']['vc_models'] = []  # Add .pth paths here if you have RVC models
config['codec_compression']['enabled'] = True
config['output']['base_dir'] = OUTPUT_PATH

with open("config.yaml", "w") as f:
    yaml.dump(config, f)

# Symlink Drive data
if not os.path.exists("mozilla_cv_data"):
    os.makedirs("mozilla_cv_data/cv-corpus-24.0-2025-12-05", exist_ok=True)
    if os.path.exists(DRIVE_DATA_PATH):
        !ln -s "{DRIVE_DATA_PATH}" "mozilla_cv_data/cv-corpus-24.0-2025-12-05/en"
        print("✅ Symlinked Drive data")
    else:
        print("⚠️ Drive path not found!")

print(f"Output: {OUTPUT_PATH}")

## 3️⃣ Generate Audio Dataset

In [ ]:
# Run the audio pipeline
!python run_pipeline.py

## 4️⃣ Extract Features (CQT or LFCC)

In [ ]:
# Choose feature type: "cqt" or "lfcc"
FEATURE_TYPE = "cqt"

!python -m pipeline.features.extract_features --type {FEATURE_TYPE}

print(f"✅ Features extracted! Type: {FEATURE_TYPE}")